In [5]:
import os
import sys
print(sys.path) 

# sys.path.insert(0, "/mnt/nas/pritish/CMUVERA_IR_ref")
sys.path.insert(0, "/mnt/dgx/pritish/CMUVERA_IR_ref")
sys.path.insert(0, "/raid/infolab/pritish/CMUVERA_IR_ref")
print(sys.path)

['/raid/infolab/pritish/CMUVERA_IR_ref', '/mnt/dgx/pritish/CMUVERA_IR_ref', '/raid/infolab/pritish/CMUVERA_IR_ref', '/mnt/dgx/pritish/CMUVERA_IR_ref', '', '/mnt/home/pritish/.local/lib/python3.11/site-packages', '/mnt/nas/pritish', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python311.zip', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/lib-dynload', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages', '/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages/setuptools/_vendor']
['/raid/infolab/pritish/CMUVERA_IR_ref', '/mnt/dgx/pritish/CMUVERA_IR_ref', '/raid/infolab/pritish/CMUVERA_IR_ref', '/mnt/dgx/pritish/CMUVERA_IR_ref', '/raid/infolab/pritish/CMUVERA_IR_ref', '/mnt/dgx/pritish/CMUVERA_IR_ref', '', '/mnt/home/pritish/.local/lib/python3.11/site-packages', '/mnt/nas/pritish', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/l

In [6]:
os.chdir("/mnt/dgx/data/pritish/CMUVERA_IR_ref")

In [7]:
import torch
import os
import numpy as np
from omegaconf import OmegaConf
import time
import logging
from src.utils import set_seed,set_seed_from_checkpoint, load, save
from tqdm import tqdm
import random
from numpy.random import default_rng
import submodlib
import pickle

from src.dataloader import get_dataloader
from src.embedder import ColBERTEmbedder

from src.utils import partial_chamfer_sim_batched_with_rerank
import torch.multiprocessing as mp

# 1) Make sure the env var is set *inside* Python too
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2")

# 2) Turn on PyTorch’s deterministic‐only mode
torch.use_deterministic_algorithms(True)

from loguru import logger

/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages/beir/util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [8]:
%load_ext autoreload
%autoreload 1

%aimport src.endtoend
%aimport make_plots

In [9]:
config = {
    "k": 15,
    "method": "v0",
    "bucket_size": 0,
    "dataset_name": "msmarco",
    "mv_type": 'colbertv2-plaid',
    "mode": "disk"
}

In [10]:
def make_config(config):
    sys.argv = ["", f"k={config['k']}", f"method={config['method']}",
                f"baseline.bucket_size={config['bucket_size']}",
                f"data.dataset_name={config['dataset_name']}",
                f"embedder.mv_type={config['mv_type']}",
                f"embedder.mode={config.get('mode', 'mem')}",
                "baseline.distributed_search=False"
    ]
    print(sys.argv)
    
    cli_conf = OmegaConf.from_cli()
    file_config = OmegaConf.load("configs/config.yaml")

    conf = OmegaConf.merge(file_config, cli_conf)

    return conf

In [11]:
# os.chdir("CMUVERA_IR_ref")

In [12]:
conf = make_config(config)

['', 'k=15', 'method=v0', 'baseline.bucket_size=0', 'data.dataset_name=msmarco', 'embedder.mv_type=colbertv2-plaid', 'embedder.mode=disk', 'baseline.distributed_search=False']


In [13]:
def get_method(config):
    name = config.method
    if name == "v0":
        return src.endtoend.GreedyBaseline_v0(config)
    elif name=="sml":
        return src.endtoend.GreedyBaseline_submodlib(config)
    else:
        raise ValueError(f"Unknown method: {name}")

In [14]:
retriever = get_method(conf)
retriever.run()

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8841823/8841823 [01:31<00:00, 97154.13it/s]


<function ColBERTEmbedder.embed_full_dataset.<locals>.<lambda> at 0x7efcb8a5ce00>
./experiments/msmarco/BERT


/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: anthropological definition of environment, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1, 28395,  6210,  1997,  4044,   102,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')



/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()


./experiments/msmarco/BERT/corpus/compressed_128/batch_0.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_1.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_2.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_3.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_4.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_5.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_6.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_7.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_8.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_9.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_10.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_11.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_12.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_13.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/batch_14.0.pkl
./experiments/msmarco/BERT/corpus/compressed_128/b

FileNotFoundError: [Errno 2] No such file or directory: './docid_to_batchinfo/docid_to_batchinfo_msmarco.json'

In [ ]:
with open(f"./pickles/results/greedy_base_0_128_k15_{config['dataset_name']}_bf.pkl", "rb") as f:
    results = pickle.load(f)

In [ ]:
results[0]

[(2716.0, 11.888933181762695),
 (1858.0, 15.326839447021484),
 (2660.0, 17.431148529052734),
 (2022.0, 18.88239288330078),
 (1853.0, 19.841400146484375),
 (190.0, 20.444477081298828),
 (1321.0, 20.736717224121094),
 (916.0, 20.99953842163086),
 (4404.0, 21.165863037109375),
 (1421.0, 21.295223236083984),
 (1632.0, 21.384984970092773),
 (2967.0, 21.444133758544922),
 (1864.0, 21.49582862854004),
 (4658.0, 21.539615631103516),
 (2495.0, 21.570064544677734)]

In [ ]:
len(results)

300